In [8]:
import pandas as pd

# ILOŚĆ KONTAKTÓW DLA CELL

# Wczytanie pliku CSV z odpowiednim separatorem
file_path = "contact_data_2000.csv" 
df = pd.read_csv(file_path, sep=",")
print(df.head())
# Liczenie liczby kontaktów dla każdego cellid
contact_counts = df['cellid'].value_counts().reset_index()
contact_counts.columns = ['cellid', 'contact_count']

# Podgląd wyników
print(contact_counts.head())


      chr1     start1       end1     chr2     start2       end2  \
0  chr13-M   74316813   74316959  chr13-M   72727004   72727154   
1   chr1-M   79322530   79322563  chr12-M    4538118    4538268   
2   chr2-M   75633331   75633491   chr8-P  125695812  125695962   
3   chr7-P  136324163  136324313   chr7-P  136352442  136352592   
4   chr6-M   49253365   49253515   chr6-M   49323546   49323680   

                       cellid  
0  SCG0088_TTTAACCTCAGCCAAT-1  
1  SCG0088_TATAGGTGTCCCGGAA-1  
2  SCG0088_CGTTAACAGTACCGCA-1  
3  SCG0088_TTTAACCTCAGCCAAT-1  
4  SCG0088_CGTTAACAGTACCGCA-1  
                       cellid  contact_count
0  SCG0092_GCTGGATGTCTAACCT-1          22890
1  SCG0089_GAGAAACGTGTTTGCT-1          21763
2  SCG0093_TGTTGTAAGTTTGCGG-1          18195
3  SCG0090_CAGCATTAGACTATTG-1          18031
4  SCG0091_TTTCTCACATTGCGGT-1          17661


In [3]:
file_path1 = "train_data_2000.csv" 
train = pd.read_csv(file_path1, sep=",")
file_path2 = "test_data_2000.csv" 
test = pd.read_csv(file_path2, sep=",")
test.head()
train.head()

,cellid,phase,order_within_phase,order
0,SCG0088_CTATGAGGTACCGGAT-1,G1,0,0
1,SCG0088_GCTAAGCGTATTGGTG-1,G1,0,0
2,SCG0089_TCCATTGTCTGTAAGC-1,G1,0,0
3,SCG0092_GTTTATCTCATGCTAA-1,G1,0,0
4,SCG0092_AACCGCTCAGCTCATA-1,G1,0,0


In [4]:
summary = df.groupby('cellid').agg(
    avg_region1_length = ('end1', lambda x: (x - df.loc[x.index, 'start1']).mean()),
    avg_region2_length = ('end2', lambda x: (x - df.loc[x.index, 'start2']).mean()),
    median_region1_length = ('end1', lambda x: (x - df.loc[x.index, 'start1']).median()),
    median_region2_length = ('end2', lambda x: (x - df.loc[x.index, 'start2']).median()),
    max_region1_length = ('end1', lambda x: (x - df.loc[x.index, 'start1']).max()),
    max_region2_length = ('end2', lambda x: (x - df.loc[x.index, 'start2']).max()),
    same_chr_contacts = ('chr1', lambda x: (x == df.loc[x.index, 'chr2']).sum()),
    total_contacts = ('cellid', 'count')
).reset_index()

summary = summary.merge(train[['cellid', 'phase']], on='cellid', how='left')

print(summary.head())

                       cellid  avg_region1_length  avg_region2_length  \
0  SCG0088_AAAGGACGTTAACGGC-1           89.425013          117.117021   
1  SCG0088_AAATCCGGTGACATAT-1           80.616243          112.613549   
2  SCG0088_AACAGCAAGACAGGCG-1           82.671569          115.306781   
3  SCG0088_AACATCATCAGGTTTA-1           87.414566          116.915349   
4  SCG0088_AACCTTAAGCTGCACA-1           80.333188          109.388066   

   median_region1_length  median_region2_length  max_region1_length  \
0                   82.0                  148.0                 204   
1                   57.0                  148.0                 172   
2                   57.0                  150.0                 168   
3                   70.0                  149.0                 177   
4                   60.0                  141.0                 170   

   max_region2_length  same_chr_contacts  total_contacts phase  
0                 182               3179            3854    G1  
1   

In [5]:
df['region1_length'] = df['end1'] - df['start1']
df['region2_length'] = df['end2'] - df['start2']
df['symmetry'] = abs(df['region1_length'] - df['region2_length'])
df['inter_chr'] = df['chr1'] != df['chr2']

# Sproszczona nazwa chromosomów
df['chr1_clean'] = df['chr1'].str.extract(r'(chr[\dXY]+)')

# Najczestsza chromosoma
top_chr = df['chr1_clean'].value_counts().idxmax()
df['top_chr1'] = df['chr1_clean'] == top_chr

df['same_chr'] = df['chr1'] == df['chr2']

extra_features = df.groupby('cellid').agg(
    percent_same_chr = ('same_chr', 'mean'),
    avg_symmetry = ('symmetry', 'mean'), 
    inter_chr_ratio = ('inter_chr', 'mean'), #Często wzrasta w fazie S lub G2
    std_region1_length = ('region1_length', 'std'), #Może różnić się między fazami ze względu na organizację chromatyny
    std_region2_length = ('region2_length', 'std'),
    total_region1_length = ('region1_length', 'sum'), #Ocenia ogólną "aktywność" regionu1
    total_region2_length = ('region2_length', 'sum'),
    top_chr1_ratio = ('top_chr1', 'mean') #Może wskazywać, że niektóre chromosomy są bardziej aktywne w konkretnych fazach
).reset_index()

summary = summary.merge(extra_features, on='cellid', how='left')

print(summary.head())

                       cellid  avg_region1_length  avg_region2_length  \
0  SCG0088_AAAGGACGTTAACGGC-1           89.425013          117.117021   
1  SCG0088_AAATCCGGTGACATAT-1           80.616243          112.613549   
2  SCG0088_AACAGCAAGACAGGCG-1           82.671569          115.306781   
3  SCG0088_AACATCATCAGGTTTA-1           87.414566          116.915349   
4  SCG0088_AACCTTAAGCTGCACA-1           80.333188          109.388066   

   median_region1_length  median_region2_length  max_region1_length  \
0                   82.0                  148.0                 204   
1                   57.0                  148.0                 172   
2                   57.0                  150.0                 168   
3                   70.0                  149.0                 177   
4                   60.0                  141.0                 170   

   max_region2_length  same_chr_contacts  total_contacts phase  \
0                 182               3179            3854    G1   
1 

In [6]:
summary = summary.merge(extra_features, on='cellid', how='left')

print(summary.head())

                       cellid  avg_region1_length  avg_region2_length  \
0  SCG0088_AAAGGACGTTAACGGC-1           89.425013          117.117021   
1  SCG0088_AAATCCGGTGACATAT-1           80.616243          112.613549   
2  SCG0088_AACAGCAAGACAGGCG-1           82.671569          115.306781   
3  SCG0088_AACATCATCAGGTTTA-1           87.414566          116.915349   
4  SCG0088_AACCTTAAGCTGCACA-1           80.333188          109.388066   

   median_region1_length  median_region2_length  max_region1_length  \
0                   82.0                  148.0                 204   
1                   57.0                  148.0                 172   
2                   57.0                  150.0                 168   
3                   70.0                  149.0                 177   
4                   60.0                  141.0                 170   

   max_region2_length  same_chr_contacts  total_contacts phase  ...  \
0                 182               3179            3854    G1 

In [7]:
from sklearn.preprocessing import MinMaxScaler

# grupowanie po cellid i obliczanie statystyk
activity_features = df.groupby('cellid').agg(
    total_contacts=('cellid', 'count'),
    avg_length_diff=('length_diff', 'mean'),
    std_region1_length=('region1_length', 'std'),
    std_region2_length=('region2_length', 'std')
).reset_index()

# normalizacja cech (żeby wartości były porównywalne)
scaler = MinMaxScaler()
activity_features[['total_contacts', 'avg_length_diff', 'std_region1_length', 'std_region2_length']] = \
    scaler.fit_transform(activity_features[['total_contacts', 'avg_length_diff', 'std_region1_length', 'std_region2_length']])

# obliczanie pseudowskaźnika aktywności jako średnia ważona cech
activity_features['activity_index'] = (
    activity_features['total_contacts'] * 0.5 +  # największy wpływ
    activity_features['avg_length_diff'] * 0.2 +
    activity_features['std_region1_length'] * 0.15 +
    activity_features['std_region2_length'] * 0.15
)

activity_features = activity_features.sort_values(by='activity_index', ascending=False)

print(activity_features.head(10))

print(activity_features.tail(10))

KeyError: "Column(s) ['length_diff'] do not exist"

In [ ]:
import numpy as np
df["distance"] = np.abs(df["start1"] - df["start2"])

In [ ]:
s = 0.2 
df['log2_distance'] = df['distance'].apply(lambda x: np.log2(x) if x > 0 else np.nan)

df["CDD"] = np.floor((df["log2_distance"] + s) / s)


In [ ]:
extra_features = df.groupby('cellid').agg(
    avg_CDD = ('CDD', 'mean'),
    min_CDD = ('CDD', 'min'),
    max_CDD = ('CDD', 'max')
)

summary = summary.merge(extra_features, on='cellid', how='left')

print(summary.head())